# coerce-float-arg-to-array — faded example 1: coerce_divisor: promote a scalar divisor to a 0-D tensor

> Practice drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `coerce-float-arg-to-array`. The last cell reports your progress on the `Backprop: Coerce float arg to array` subtopic back to Delta Drills.

**Most of the code is already written — complete the one blanked step**, run the test to check it, then run the last cell to record your progress.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """Minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries optional `.recipe`,
    `.requires_grad`, and `.grad` (the accumulated gradient at leaves)."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: Coerce float arg to array` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`coerce-float-arg-to-array`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "coerce-float-arg-to-array"
DD_SUBTOPIC = "Backprop: Coerce float arg to array"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

When wrapping `divide(t, denom)` for autograd, a Python scalar `denom` (e.g. `divide(x, 4.0)`) must be promoted to a 0-D tensor so the Recipe stores a tensor and back fns avoid type checks. The trap is `bool` (a subclass of `int`) which is a control flag and must pass through; tensors and other objects also pass through unchanged.

## Faded exercise 1

Implement `coerce_divisor(arg)`. A real `int` or `float` becomes `t.tensor(float(arg))` (0-D float32). A `bool` passes through (it is a flag, not a scalar). Anything else (`torch.Tensor`, tuple, `None`, ...) passes through unchanged. Complete the one blanked line that produces the coerced 0-D tensor for the real-scalar case.

**Your task:** complete the one blanked step in the code cell below. The surrounding code, function signatures, and variable names are given — work out the missing expression yourself, then run the test.

In [ ]:
def coerce_divisor(arg):
    if isinstance(arg, bool):
        return arg
    if isinstance(arg, (int, float)):
        coerced = None  # TODO: fill in this step — read the prompt cell above
        return coerced
    return arg

val = coerce_divisor(4.0)
flag = coerce_divisor(True)
kept = coerce_divisor(t.ones(2))
print(val, val.dtype if hasattr(val, 'dtype') else None)
print(flag, type(flag).__name__)
print(kept.shape)


def _test():
    r = coerce_divisor(4.0)
    assert isinstance(r, t.Tensor), "float must coerce to a tensor"
    assert r.ndim == 0, "must be 0-D"
    assert r.dtype == t.float32, "must be float32"
    assert float(r) == 4.0
    ri = coerce_divisor(5)
    assert isinstance(ri, t.Tensor) and float(ri) == 5.0 and ri.dtype == t.float32
    b = coerce_divisor(True)
    assert b is True and isinstance(b, bool), "bool must pass through unchanged"
    ten = t.ones(2)
    assert coerce_divisor(ten) is ten, "tensor must pass through (identity)"
    assert coerce_divisor(None) is None


try:
    _test()
    _dd_passed.add('faded1')
    print('[Delta Drills] faded1 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report your progress

Run the cell below to send your progress to Delta Drills. It only counts if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
def coerce_divisor(arg):
    if isinstance(arg, bool):
        return arg
    if isinstance(arg, (int, float)):
        coerced = t.tensor(float(arg))  # wrap real scalar as 0-D float32 tensor
        return coerced
    return arg

val = coerce_divisor(4.0)
flag = coerce_divisor(True)
kept = coerce_divisor(t.ones(2))
print(val, val.dtype if hasattr(val, 'dtype') else None)
print(flag, type(flag).__name__)
print(kept.shape)
```
</details>